 Merge, Clean, Deduplicate, and Prepare ArXiv Labels

This notebook harmonizes `applied`, `theory`, and `survey` CSV files into a single labeled dataset.
It performs cleaning, duplicate/conflict handling, and creates stratified
train/validation/test splits for downstream modeling.


In [ ]:
import hashlib
import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_STATE = 42

DATASET_DIR = Path("Dataset")
DATA_DIR = Path("data")

DATA_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)


In [2]:
def build_harmonized_frame(
    frame: pd.DataFrame,
    column_map: dict,
    source_dataset: str,
    fixed_label: str | None = None,
) -> pd.DataFrame:
    out = pd.DataFrame()
    for target_col, source_col in column_map.items():
        out[target_col] = frame[source_col] if source_col in frame.columns else np.nan
    if fixed_label is not None:
        out["label"] = fixed_label
    out["source_dataset"] = source_dataset
    return out


canonical_columns = [
    "label",
    "title",
    "abstract",
    "arxiv_id",
    "primary_category",
    "categories",
    "published",
    "arxiv_url",
    "topic",
    "list_name",
    "found_in_arxiv_api",
    "source_dataset",
]

applied_raw = pd.read_csv(DATASET_DIR / "applied_papers_1000.csv")
theory_raw = pd.read_csv(DATASET_DIR / "theory_papers_1000.csv")
survey_raw = pd.read_csv(DATASET_DIR / "survey_papers_901.csv")

applied_map = {
    "label": "label",
    "title": "title",
    "abstract": "abstract",
    "arxiv_id": "arxiv_id",
    "primary_category": "primary_category",
    "categories": "categories",
    "published": "published",
    "arxiv_url": "arxiv_url",
    "topic": "topic",
    "list_name": "list_name",
    "found_in_arxiv_api": "found_in_arxiv_api",
}

theory_map = dict(applied_map)

survey_map = {
    "label": "label",
    "title": "arxiv_title",
    "abstract": "arxiv_abstract",
    "arxiv_id": "arxiv_id",
    "primary_category": "primary_category",
    "categories": "categories",
    "published": "published",
    "arxiv_url": "arxiv_url",
    "topic": "topic",
    "list_name": "list_name",
    "found_in_arxiv_api": "found_in_arxiv_api",
}

applied_df = build_harmonized_frame(applied_raw, applied_map, source_dataset="applied")
theory_df = build_harmonized_frame(theory_raw, theory_map, source_dataset="theory")
survey_df = build_harmonized_frame(
    survey_raw,
    survey_map,
    source_dataset="survey",
    fixed_label="SURVEY",
)

merged_df = pd.concat([applied_df, theory_df, survey_df], ignore_index=True)
merged_df = merged_df[canonical_columns]

print("Merged shape:", merged_df.shape)
print("Columns:", list(merged_df.columns))
merged_df.head(3)


Merged shape: (2901, 12)
Columns: ['label', 'title', 'abstract', 'arxiv_id', 'primary_category', 'categories', 'published', 'arxiv_url', 'topic', 'list_name', 'found_in_arxiv_api', 'source_dataset']


,label,title,abstract,arxiv_id,primary_category,categories,published,arxiv_url,topic,list_name,found_in_arxiv_api,source_dataset
0,APPLIED,Towards Generalizable Robotic Manipulation in ...,Vision-Language-Action (VLA) models excel in s...,2603.15620v1,cs.CV,cs.CV cs.RO,2026-03-16T17:59:57Z,http://arxiv.org/abs/2603.15620v1,NaN,NaN,NaN,applied
1,APPLIED,Mixture-of-Depths Attention,Scaling depth is a key driver for large langua...,2603.15619v1,cs.CL,cs.AI cs.CL,2026-03-16T17:59:55Z,http://arxiv.org/abs/2603.15619v1,NaN,NaN,NaN,applied
2,APPLIED,Look Before Acting: Enhancing Vision Foundatio...,Vision-Language-Action (VLA) models have recen...,2603.15618v1,cs.CV,cs.CV,2026-03-16T17:59:54Z,http://arxiv.org/abs/2603.15618v1,NaN,NaN,NaN,applied


In [3]:
def normalize_text(value: object):
    if pd.isna(value):
        return np.nan
    text = re.sub(r"\s+", " ", str(value).replace("\n", " ")).strip()
    if text == "":
        return np.nan
    return text


clean_df = merged_df.copy()

for col in ["label", "title", "abstract", "primary_category", "categories", "topic", "list_name"]:
    if col in clean_df.columns:
        clean_df[col] = clean_df[col].apply(normalize_text)

before_drop_missing = len(clean_df)
clean_df = clean_df.dropna(subset=["title", "abstract"]).copy()
dropped_missing = before_drop_missing - len(clean_df)

clean_df["label"] = clean_df["label"].fillna("UNKNOWN")
clean_df["model_text"] = clean_df["title"] + " " + clean_df["title"] + " " + clean_df["abstract"]

clean_df["title_norm"] = clean_df["title"].str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
clean_df["abstract_norm"] = clean_df["abstract"].str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
clean_df["text_fingerprint"] = (
    clean_df["title_norm"] + " || " + clean_df["abstract_norm"]
).map(lambda x: hashlib.md5(x.encode("utf-8")).hexdigest())

print(f"Dropped rows with missing title/abstract: {dropped_missing}")
print("Rows after missing-text cleaning:", len(clean_df))


Dropped rows with missing title/abstract: 0
Rows after missing-text cleaning: 2901


In [4]:
non_null_ids = clean_df.dropna(subset=["arxiv_id"]).copy()
label_per_id = non_null_ids.groupby("arxiv_id")["label"].nunique()
conflicting_ids = label_per_id[label_per_id > 1].index

conflicting_rows = clean_df[clean_df["arxiv_id"].isin(conflicting_ids)].copy()
after_conflict_df = clean_df[~clean_df["arxiv_id"].isin(conflicting_ids)].copy()

same_id_mask = after_conflict_df["arxiv_id"].notna() & after_conflict_df.duplicated(subset=["arxiv_id"], keep="first")
same_id_removed = after_conflict_df[same_id_mask].copy()
after_id_df = after_conflict_df.loc[~same_id_mask].copy()

same_text_mask = after_id_df.duplicated(subset=["text_fingerprint"], keep="first")
text_removed = after_id_df[same_text_mask].copy()
cleaned_df = after_id_df.loc[~same_text_mask].copy()

dedup_audit = pd.DataFrame(
    [
        {"metric": "rows_after_missing_text_drop", "value": int(len(clean_df))},
        {"metric": "conflicting_ids_count", "value": int(len(conflicting_ids))},
        {"metric": "rows_removed_conflicting_labels", "value": int(len(conflicting_rows))},
        {"metric": "rows_removed_same_id", "value": int(len(same_id_removed))},
        {"metric": "rows_removed_same_text", "value": int(len(text_removed))},
        {"metric": "rows_after_dedup", "value": int(len(cleaned_df))},
    ]
)

conflicting_rows.to_csv(DATA_DIR / "dedup_conflicting_rows.csv", index=False)
same_id_removed.to_csv(DATA_DIR / "dedup_same_id_removed.csv", index=False)
text_removed.to_csv(DATA_DIR / "dedup_text_removed.csv", index=False)
dedup_audit.to_csv(DATA_DIR / "dedup_audit_summary.csv", index=False)

print(dedup_audit)
print()
print("Label counts after dedup:")
print(cleaned_df["label"].value_counts())


                            metric  value
0     rows_after_missing_text_drop   2901
1            conflicting_ids_count      2
2  rows_removed_conflicting_labels      4
3             rows_removed_same_id     42
4           rows_removed_same_text      0
5                 rows_after_dedup   2855

Label counts after dedup:
label
APPLIED        998
THEORETICAL    998
SURVEY         859
Name: count, dtype: int64


In [5]:
missingness = (
    cleaned_df.isna()
    .mean()
    .sort_values(ascending=False)
    .rename("missing_fraction")
    .reset_index(name="missing_fraction")
    .rename(columns={"index": "column"})
)
label_counts = cleaned_df["label"].value_counts().sort_values(ascending=False).rename("count")
primary_counts = (
    cleaned_df["primary_category"]
    .fillna("<missing>")
    .value_counts()
    .head(20)
    .rename("count")
)

label_counts.to_csv(DATA_DIR / "eda_label_counts.csv", header=True)
missingness.to_csv(DATA_DIR / "eda_missingness.csv", index=False)
primary_counts.to_csv(DATA_DIR / "eda_primary_category_counts.csv", header=True)

cleaned_df["title_length"] = cleaned_df["title"].str.split().str.len()
cleaned_df["abstract_length"] = cleaned_df["abstract"].str.split().str.len()
cleaned_df["published_year"] = pd.to_datetime(cleaned_df["published"], errors="coerce").dt.year

length_summary = (
    cleaned_df.groupby("label")[["title_length", "abstract_length"]]
    .agg(["mean", "median"])
    .round(2)
)

yearly = (
    cleaned_df.dropna(subset=["published_year"])
    .groupby(["published_year", "label"])
    .size()
    .unstack(fill_value=0)
    .sort_index()
)

print("Saved EDA summary tables in data/ (no plots generated).")
print("\nMissingness (top 10):")
print(missingness.head(10))
print("\nLabel counts:")
print(label_counts)
print("\nLength summary by label (mean/median):")
print(length_summary)
if yearly.empty:
    print("\nNo valid published_year values found.")
else:
    print("\nYearly counts shape:", yearly.shape)

Saved EDA summary tables in data/ (no plots generated).

Missingness (top 10):
               column  missing_fraction
0           list_name          0.699124
1  found_in_arxiv_api          0.699124
2               topic          0.699124
3               label          0.000000
4    primary_category          0.000000
5               title          0.000000
6            abstract          0.000000
7            arxiv_id          0.000000
8           arxiv_url          0.000000
9           published          0.000000

Label counts:
label
APPLIED        998
THEORETICAL    998
SURVEY         859
Name: count, dtype: int64

Length summary by label (mean/median):
            title_length        abstract_length       
                    mean median            mean median
label                                                 
APPLIED            10.28   10.0          181.07  180.0
SURVEY              9.15    9.0          160.07  160.0
THEORETICAL         8.49    8.0          172.25  169.5

Yearly

In [6]:
TOKEN_PATTERN = re.compile(r"[A-Za-z][A-Za-z0-9_-]+")


def tokenize(text: str) -> list[str]:
    tokens = [t.lower() for t in TOKEN_PATTERN.findall(text)]
    return [t for t in tokens if len(t) > 2]


token_rows = []
for label, group in cleaned_df.groupby("label"):
    counter = Counter()
    for text in group["model_text"].fillna(""):
        counter.update(tokenize(text))
    for token, count in counter.most_common(30):
        token_rows.append({"label": label, "token": token, "count": int(count)})

token_df = pd.DataFrame(token_rows)
token_df.to_csv(DATA_DIR / "eda_top_tokens_per_label.csv", index=False)
token_df.head(20)


,label,token,count
0,APPLIED,and,6622
1,APPLIED,the,5587
2,APPLIED,for,3007
3,APPLIED,that,2201
4,APPLIED,with,1776
5,APPLIED,this,1292
6,APPLIED,models,1148
7,APPLIED,from,1055
8,APPLIED,model,899
9,APPLIED,learning,792


In [8]:
cleaned_output = DATA_DIR / "combined_papers_cleaned.csv"
cleaned_df.to_csv(cleaned_output, index=False)

print(f"Saved cleaned dataset to {cleaned_output}")
print("Final cleaned shape:", cleaned_df.shape)
print("Final label counts:")
print(cleaned_df["label"].value_counts())


Saved cleaned dataset to data/combined_papers_cleaned.csv
Final cleaned shape: (2855, 19)
Final label counts:
label
APPLIED        998
THEORETICAL    998
SURVEY         859
Name: count, dtype: int64
